In [4]:
import requests

url = "https://gist.githubusercontent.com/HenryQW/c2a353415a3ce88f829c3b1199ee3c17/raw/NYC%20TLC%20taxi%20zones.geojson"
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})

print("Content-Length заголовок:", response.headers.get('Content-Length'))
print("Реально получено байт:", len(response.content))


Content-Length заголовок: 921600
Реально получено байт: 921600


In [5]:
import requests

url = "https://gist.githubusercontent.com/HenryQW/c2a353415a3ce88f829c3b1199ee3c17/raw/NYC%20TLC%20taxi%20zones.geojson"

with requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, stream=True) as r:
    r.raise_for_status()
    with open('taxi_zones.geojson', 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

import os
print("Размер сохранённого файла:", os.path.getsize('taxi_zones.geojson'), "байт")

Размер сохранённого файла: 921600 байт


In [6]:
import requests

url = "https://data.cityofnewyork.us/api/geospatial/8meu-9t5y?method=export&format=GeoJSON"
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()

print("Размер ответа:", len(response.content), "байт")

data = response.json()
print(f"Загружено зон: {len(data['features'])}")

Размер ответа: 3885973 байт
Загружено зон: 263


In [7]:
import requests
import json
import csv

# Шаг 1. Загрузка данных с официального портала NYC Open Data
url = "https://data.cityofnewyork.us/api/geospatial/8meu-9t5y?method=export&format=GeoJSON"

print("Загружаю файл...")
response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
response.raise_for_status()
print(f"Размер ответа: {len(response.content)} байт")

data = response.json()
print(f"Загружено зон: {len(data['features'])}")

# Шаг 2. Смотрим на структуру полей первой зоны, чтобы понять точные названия
first_props = data['features'][0]['properties']
print("\nДоступные поля:", list(first_props.keys()))
print("\nПример значений:", first_props)

# Шаг 3. Функция для поиска нужного поля независимо от регистра/названия
def find_field(props, candidates):
    """Ищет поле среди возможных вариантов названия (без учёта регистра)"""
    props_lower = {k.lower(): k for k in props.keys()}
    for candidate in candidates:
        if candidate.lower() in props_lower:
            return props_lower[candidate.lower()]
    return None

# Определяем реальные названия полей на этом источнике
sample_props = data['features'][0]['properties']
id_field = find_field(sample_props, ['LocationID', 'location_id', 'objectid', 'OBJECTID'])
zone_field = find_field(sample_props, ['zone', 'Zone', 'zone_name'])
borough_field = find_field(sample_props, ['borough', 'Borough'])

print(f"\nНайденные поля: id={id_field}, zone={zone_field}, borough={borough_field}")

if not all([id_field, zone_field, borough_field]):
    print(" Не все поля найдены автоматически — проверь список полей выше и укажи названия вручную")

# Шаг 4. Функция перестановки координат [lon, lat] -> [lat, lon]
def swap_coords(coords):
    if isinstance(coords[0], (int, float)):
        return [coords[1], coords[0]]
    return [swap_coords(c) for c in coords]

# Шаг 5. Конвертация всех зон
rows = []
errors = 0
for feature in data['features']:
    try:
        props = feature['properties']
        geom = feature['geometry']
        if geom is None:
            errors += 1
            continue
        swapped = swap_coords(geom['coordinates'])
        coord_str = json.dumps(swapped)
        rows.append({
            'LocationID': props.get(id_field),
            'zone': props.get(zone_field),
            'borough': props.get(borough_field),
            'coordinates': coord_str
        })
    except Exception as e:
        errors += 1
        print(f"Ошибка на зоне: {e}")

print(f"\nУспешно сконвертировано: {len(rows)} зон, ошибок: {errors}")

# Шаг 6. Сохранение в CSV
output_file = 'taxi_zones_datalens.csv'
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['LocationID', 'zone', 'borough', 'coordinates'])
    writer.writeheader()
    writer.writerows(rows)

print(f"\n Готово! Файл сохранён: {output_file}")

Загружаю файл...
Размер ответа: 3885973 байт
Загружено зон: 263

Доступные поля: ['shape_area', 'locationid', 'shape_leng', 'zone', 'borough']

Пример значений: {'shape_area': '0.0007823067885', 'locationid': '1', 'shape_leng': '0.116357453189', 'zone': 'Newark Airport', 'borough': 'EWR'}

Найденные поля: id=locationid, zone=zone, borough=borough

Успешно сконвертировано: 263 зон, ошибок: 0

 Готово! Файл сохранён: taxi_zones_datalens.csv


In [8]:
def swap_coords(coords):
    if isinstance(coords[0], (int, float)):
        return [coords[1], coords[0]]
    return [swap_coords(c) for c in coords]

rows = []
for feature in data['features']:
    props = feature['properties']
    geom = feature['geometry']
    if geom is None:
        continue
    
    geom_type = geom['type']
    coords = geom['coordinates']
    
    if geom_type == 'Polygon':
        # Обычный полигон - используем как есть
        polygons = [coords]
    elif geom_type == 'MultiPolygon':
        # Мультиполигон - разворачиваем в список отдельных полигонов
        polygons = coords
    else:
        continue
    
    for poly in polygons:
        swapped = swap_coords(poly)
        coord_str = json.dumps(swapped)
        rows.append({
            'LocationID': props.get(id_field),
            'zone': props.get(zone_field),
            'borough': props.get(borough_field),
            'coordinates': coord_str
        })

with open('taxi_zones_datalens.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['LocationID', 'zone', 'borough', 'coordinates'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Готово: {len(rows)} строк (некоторые зоны могли дать несколько строк, если были MultiPolygon)")

Готово: 354 строк (некоторые зоны могли дать несколько строк, если были MultiPolygon)


In [ ]:
def swap_coords(coords):
    if isinstance(coords[0], (int, float)):
        return [coords[1], coords[0]]
    return [swap_coords(c) for c in coords]

rows = []
for feature in data['features']:
    props = feature['properties']
    geom = feature['geometry']
    swapped = swap_coords(geom['coordinates'])
    coord_str = json.dumps(swapped)
    rows.append({
        'LocationID': props['LocationID'],
        'zone': props['zone'],
        'borough': props['borough'],
        'coordinates': coord_str
    })

import csv
with open('taxi_zones_datalens.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['LocationID', 'zone', 'borough', 'coordinates'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Готово: {len(rows)} зон")